[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safe-t-ai/safe-t-ai.github.io/blob/main/notebooks/04_test3_infrastructure.ipynb)

# Test 3: Infrastructure Recommendation Audit

Runs the `InfrastructureRecommendationAuditor` to simulate AI vs. need-based budget allocation
across Durham census tracts, exports audit artifacts, and generates static frontend files.

## 1. Install dependencies

In [ ]:
!pip install -q geopandas

## 2. Bootstrap repo

In [ ]:
import subprocess
import sys
from pathlib import Path

NOTEBOOKS_DIR = Path("/content/safe-t-ai.github.io/notebooks")
if not NOTEBOOKS_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/safe-t-ai/safe-t-ai.github.io.git",
         "/content/safe-t-ai.github.io"],
        check=True,
    )

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from colab_utils import prepare_notebook, publish_artifacts

REPO = prepare_notebook(pull_latest=True)
print(f"Repo root: {REPO}")

## 3. Load census and infrastructure data

In [ ]:
import json
import pandas as pd
import geopandas as gpd

from config import (
    INFRASTRUCTURE_DEFAULT_BUDGET, DEFAULT_RANDOM_SEED,
    RAW_DATA_DIR, SIMULATED_DATA_DIR,
)
from models.infrastructure_auditor import InfrastructureRecommendationAuditor
from utils.data_loading import load_infrastructure_data

census_file = RAW_DATA_DIR / "durham_census_tracts.geojson"
if not census_file.exists():
    raise FileNotFoundError(f"Census file not found at {census_file}. Run fetch_durham_data.py first.")

census_gdf = gpd.read_file(census_file)
print(f"Loaded {len(census_gdf)} census tracts")

infrastructure_df = load_infrastructure_data()
print(f"Loaded infrastructure scores for {len(infrastructure_df)} tracts")

## 4. Run simulation

In [ ]:
print(f"Initializing auditor with ${INFRASTRUCTURE_DEFAULT_BUDGET:,} total budget")
auditor = InfrastructureRecommendationAuditor(census_gdf, infrastructure_df)

print("Simulating danger scores...")
danger_scores = auditor.simulate_danger_scores(seed=DEFAULT_RANDOM_SEED)
print(f"  Average danger score: {danger_scores['danger_score'].mean():.1f} crashes/10k/year")
print(f"  Range: {danger_scores['danger_score'].min():.1f} - {danger_scores['danger_score'].max():.1f}")

income_danger_corr = danger_scores["median_income"].corr(danger_scores["danger_score"])
print(f"  Income-Danger correlation: {income_danger_corr:.3f}")

print("\nSimulating AI recommendations (biased toward high-income)...")
ai_recs = auditor.simulate_ai_recommendations(bias_strength=0.6, seed=DEFAULT_RANDOM_SEED)
print(f"  Projects: {len(ai_recs)}, Budget allocated: ${ai_recs['cost'].sum():,}")

print("\nSimulating need-based recommendations (equitable baseline)...")
need_recs = auditor.simulate_need_based_recommendations(seed=DEFAULT_RANDOM_SEED)
print(f"  Projects: {len(need_recs)}, Budget allocated: ${need_recs['cost'].sum():,}")

## 5. Calculate equity metrics

In [ ]:
equity_metrics = auditor.calculate_equity_metrics()

ai_alloc = equity_metrics["ai_allocation"]
need_alloc = equity_metrics["need_based_allocation"]
comparison = equity_metrics["comparison"]

print("AI Allocation (biased):")
print(f"  Disparate Impact Ratio: {ai_alloc['disparate_impact_ratio']:.3f}")
print(f"  Gini Coefficient: {ai_alloc['gini_coefficient']:.3f}")

print("\nNeed-Based Allocation (equitable):")
print(f"  Disparate Impact Ratio: {need_alloc['disparate_impact_ratio']:.3f}")
print(f"  Gini Coefficient: {need_alloc['gini_coefficient']:.3f}")

print(f"\nEquity Gap: {comparison['equity_gap']:.3f}")
print(f"Gini Improvement: {comparison['gini_improvement']:.3f}")

## 6. Export infrastructure_recommendations.json

In [ ]:
SIMULATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

report = auditor.generate_report()
report["_provenance"] = {
    "data_type": "mixed",
    "real": ["infrastructure gap analysis (OpenStreetMap)"],
    "simulated": ["danger scores", "AI recommendations", "need-based recommendations"],
    "parameters": {"budget": INFRASTRUCTURE_DEFAULT_BUDGET, "seed": DEFAULT_RANDOM_SEED},
}

output_file = SIMULATED_DATA_DIR / "infrastructure_recommendations.json"
with open(output_file, "w") as f:
    json.dump(report, f, indent=2)
print(f"Exported {output_file.name}")

if report.get("findings"):
    print("\nKey findings:")
    for finding in report["findings"]:
        print(f"  - {finding}")

## 7. Generate frontend static files

In [ ]:
from utils.geospatial import geojson_to_dict, simplify_geometry

frontend_data_dir = REPO / "frontend" / "public" / "data"
frontend_data_dir.mkdir(parents=True, exist_ok=True)

def copy_json(src, dst, indent=2):
    with open(src) as f:
        data = json.load(f)
    with open(dst, "w") as f:
        json.dump(data, f, indent=indent)
    return data

# infrastructure-report.json
infrastructure_data = copy_json(output_file, frontend_data_dir / "infrastructure-report.json")
print("Wrote infrastructure-report.json")

# danger-scores.json
danger_df = census_gdf.merge(pd.DataFrame(infrastructure_data["danger_scores"]), on="tract_id")
income_col = "median_income_y" if "median_income_y" in danger_df.columns else "median_income"
danger_df["income_quintile"] = pd.qcut(
    danger_df[income_col], q=5,
    labels=["Q1 (Poorest)", "Q2", "Q3", "Q4", "Q5 (Richest)"]
)
danger_df = simplify_geometry(danger_df, tolerance=0.001)
with open(frontend_data_dir / "danger-scores.json", "w") as f:
    json.dump(geojson_to_dict(danger_df), f)
print("Wrote danger-scores.json")

# budget-allocation.json
allocation_comparison = {
    "ai_allocation": infrastructure_data["equity_metrics"]["ai_allocation"],
    "need_based_allocation": infrastructure_data["equity_metrics"]["need_based_allocation"],
    "comparison": infrastructure_data["equity_metrics"]["comparison"],
}
with open(frontend_data_dir / "budget-allocation.json", "w") as f:
    json.dump(allocation_comparison, f, indent=2)
print("Wrote budget-allocation.json")

# recommendations.json
ai_recs_df = pd.DataFrame(infrastructure_data["ai_recommendations"])
need_recs_df = pd.DataFrame(infrastructure_data["need_based_recommendations"])
ai_recs_gdf = simplify_geometry(census_gdf.merge(ai_recs_df, on="tract_id", how="inner"), tolerance=0.001)
need_recs_gdf = simplify_geometry(census_gdf.merge(need_recs_df, on="tract_id", how="inner"), tolerance=0.001)
recommendations_data = {
    "ai_recommendations": geojson_to_dict(ai_recs_gdf),
    "need_based_recommendations": geojson_to_dict(need_recs_gdf),
}
with open(frontend_data_dir / "recommendations.json", "w") as f:
    json.dump(recommendations_data, f, indent=2)
print("Wrote recommendations.json")

## 8. Publish artifacts

In [ ]:
publish_artifacts(
    [
        "backend/data/simulated/infrastructure_recommendations.json",
        "frontend/public/data/infrastructure-report.json",
        "frontend/public/data/danger-scores.json",
        "frontend/public/data/budget-allocation.json",
        "frontend/public/data/recommendations.json",
    ],
    message="data: regenerate Test 3 infrastructure",
    repo_dir=REPO,
)